# Multinomial Naive Bayes (from scratch) — Fit & Predict Tracing

This notebook traces the intermediate values produced by the `fit()` and `predict()` steps of a simple Multinomial Naive Bayes implementation.


## Imports

The notebook uses:
- `numpy` for vectorized math.
- `pandas` for building a small example count-matrix `X`.
- `scipy.sparse.issparse` to handle both dense and sparse matrices.


In [2]:
from scipy.sparse import issparse
import numpy as np 
import pandas as pd

## Toy dataset (X, y)

`X` is a small count matrix (5 samples × 4 features) and `y` contains two classes: `"spam"` and `"ham"`.  
This example is intentionally small so every intermediate array in `fit()` and `predict()` can be inspected.


In [3]:
X = pd.DataFrame(
    [
        [2, 0, 1, 0],  # sample 0
        [0, 1, 0, 3],  # sample 1
        [1, 0, 0, 1],  # sample 2
        [0, 2, 1, 0],  # sample 3
        [0, 0, 4, 1],  # sample 4
    ]
)

y = ["spam", "ham", "ham", "spam", "ham"]
issparse(X)

False

## Tracing `fit(X, y)`

The `fit()` method learns two things:

1) **Class prior**  
- Convert labels to indices (e.g., `"ham" → 0`, `"spam" → 1`) using `np.unique(..., return_inverse=True)`.  
- Count samples per class with `np.bincount(y_idx)`.  
- Compute log priors: `log_prior_ = log(class_count) - log(n_samples)`.

2) **Conditional feature probabilities**  
- For each class `c`, sum feature counts over rows that belong to that class to build `word_count[c]`.  
- Apply Laplace smoothing: `smoothed = word_count + alpha`.  
- Normalize per class: `denom = smoothed.sum(axis=1, keepdims=True)`.  
- Store log probabilities: `log_prob_ = log(smoothed / denom)`.

In the next code cell(s), print these variables to see the full training trace.


In [4]:
y = np.asarray(y)
y

array(['spam', 'ham', 'ham', 'spam', 'ham'], dtype='<U4')

In [5]:
classes_, y_idx = np.unique(y, return_inverse=True)
y_idx

array([1, 0, 0, 1, 0])

In [6]:
n_classes = len(classes_)
n_classes

2

In [7]:
n_features = X.shape[1]
n_features

4

In [8]:
class_count = np.bincount(y_idx)
class_count

array([3, 2])

In [9]:
log_prior_ = np.log(class_count) - np.log(len(y))
log_prior_

array([-0.51082562, -0.91629073])

In [10]:
word_count = np.zeros((n_classes, n_features), dtype=float)
word_count

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [11]:
for c in range(n_classes):
            rows = (y_idx == c)
            if issparse(X):
                word_count[c] = np.asarray(X[rows].sum(axis=0)).ravel()
            else:
                word_count[c] = X[rows].sum(axis=0)

In [12]:
word_count

array([[1., 1., 4., 5.],
       [2., 2., 2., 0.]])

In [13]:
smoothed = word_count + 1 #alpha = 1
smoothed

array([[2., 2., 5., 6.],
       [3., 3., 3., 1.]])

In [14]:
denom = smoothed.sum(axis=1, keepdims=True)
denom

array([[15.],
       [10.]])

In [15]:
log_prob_ = np.log(smoothed / denom)
log_prob_

array([[-2.01490302, -2.01490302, -1.09861229, -0.91629073],
       [-1.2039728 , -1.2039728 , -1.2039728 , -2.30258509]])

## Tracing `predict(X)`

Prediction scores each sample against each class:

- Compute a log-score per class using the dot product:
  - Dense: `scores = np.dot(X, log_prob_.T)`
  - Sparse: `scores = X @ log_prob_.T`
- Add class priors: `scores = scores + log_prior_`.
- Pick the highest-scoring class per row: `argmax(scores, axis=1)`.
- Map indices back to labels using `classes_`.

In the next code cell(s), display `scores_before_prior`, `scores_after_prior`, and the final predicted labels.


In [18]:
new_x = pd.DataFrame([[3,4,2,0]])

In [20]:
scores = np.dot(new_x, log_prob_.T)
scores

array([[-16.30154572, -10.83575524]])

pick the higer score

In [21]:
classes_[scores.argmax(axis=1)]

array(['spam'], dtype='<U4')